# RankSEG with SAM3

This Colab notebook demonstrates RankSEG with `facebook/sam3` image Promptable Concept Segmentation.

It covers two paths:

- SAM3 instance masks: use the official `processor.post_process_instance_segmentation(...)` baseline, then apply `RankSEG` directly to selected raw mask probabilities.
- SAM3 semantic output: use the official `processor.post_process_semantic_segmentation(...)` baseline, then use `rankseg.transformers.postprocess(...)` on `outputs.semantic_seg`.

`facebook/sam3` is gated on Hugging Face, so this notebook starts with a Hugging Face login cell. The account must have access to the model.

## 0. Setup

In [ ]:
%pip install -q uv
!uv pip install --system --no-deps "git+https://github.com/rankseg/rankseg@main"

Log in before loading `facebook/sam3`. Paste a Hugging Face token from an account with access to the gated model.

In [ ]:
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token)
else:
    notebook_login()

In [ ]:
import numpy as np
import requests
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

from rankseg import RankSEG
from rankseg.transformers import postprocess as rankseg_transformers_postprocess
from transformers import Sam3Model, Sam3Processor


def load_rgb_image(url):
    response = requests.get(url, stream=True, timeout=30)
    response.raise_for_status()
    return Image.open(response.raw).convert("RGB")


def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def show_mask(mask, ax, color=(0.12, 0.56, 1.0), alpha=0.55):
    mask = to_numpy(mask).astype(bool)
    overlay = np.zeros((*mask.shape[-2:], 4), dtype=np.float32)
    overlay[..., :3] = np.array(color, dtype=np.float32)
    overlay[..., 3] = mask.astype(np.float32) * alpha
    ax.imshow(overlay)


def show_image_with_mask(image, mask, ax, title):
    ax.imshow(image)
    show_mask(mask, ax)
    ax.set_title(title)
    ax.axis("off")


def show_binary(mask, ax, title):
    ax.imshow(to_numpy(mask), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")


device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device != "cuda":
    print("SAM3 can run on CPU, but Colab GPU is strongly recommended.")

## 1. Load SAM3 and prepare one prompt

The notebook uses one image and one text prompt so that the RankSEG hand-off is easy to inspect.

In [ ]:
checkpoint = "facebook/sam3"

processor = Sam3Processor.from_pretrained(checkpoint, token=True)
model = Sam3Model.from_pretrained(checkpoint, device_map="auto", token=True)
model.eval()

print("model device:", model.device)

In [ ]:
image_url = "http://images.cocodataset.org/val2017/000000077595.jpg"
image = load_rgb_image(image_url)
text_prompt = "ear"

plt.figure(figsize=(7, 7))
plt.imshow(image)
plt.title(f"Input image, prompt: {text_prompt!r}")
plt.axis("off");

In [ ]:
inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(model.device)
original_size = tuple(int(v) for v in inputs["original_sizes"][0].tolist())

with torch.inference_mode():
    outputs = model(**inputs)

print("original_size:", original_size)
print("pred_masks:", tuple(outputs.pred_masks.shape))
print("pred_boxes:", tuple(outputs.pred_boxes.shape))
print("pred_logits:", tuple(outputs.pred_logits.shape))
print("semantic_seg:", tuple(outputs.semantic_seg.shape))

## 2. Instance masks

SAM3 instance segmentation uses `pred_logits` and `pred_masks`. The generic RankSEG Transformers helper intentionally does not claim this full instance adapter path, so this section applies `RankSEG` directly to selected SAM3 mask probabilities.

In [ ]:
instance_threshold = 0.5
mask_threshold = 0.5
max_instance_queries = 3

official_instances = processor.post_process_instance_segmentation(
    outputs,
    threshold=instance_threshold,
    mask_threshold=mask_threshold,
    target_sizes=inputs["original_sizes"].tolist(),
)[0]

scores = outputs.pred_logits.sigmoid()[0]
if outputs.presence_logits is not None:
    scores = scores * outputs.presence_logits.sigmoid()[0].reshape(-1)[0]

query_order = torch.argsort(scores, descending=True)
selected_queries = query_order[scores[query_order] > instance_threshold][:max_instance_queries]
if selected_queries.numel() == 0:
    selected_queries = query_order[:max_instance_queries]

selected_scores = scores[selected_queries]
instance_logits = F.interpolate(
    outputs.pred_masks[:, selected_queries].float(),
    size=original_size,
    mode="bilinear",
    align_corners=False,
)
instance_probs = instance_logits.sigmoid()

threshold_instance_masks = (instance_probs > mask_threshold).to(torch.long)
rankseg_instance_masks = RankSEG(metric="iou", solver="RMA", output_mode="multilabel").predict(instance_probs)

print("official instance count:", len(official_instances["masks"]))
print("selected raw query indices:", selected_queries.detach().cpu().tolist())
print("selected scores:", [round(float(v), 4) for v in selected_scores.detach().cpu()])

In [ ]:
empty_mask = torch.zeros(original_size, dtype=torch.long)
official_mask = official_instances["masks"][0].detach().cpu() if len(official_instances["masks"]) else empty_mask
threshold_query_mask = threshold_instance_masks[0, 0].detach().cpu()
rankseg_query_mask = rankseg_instance_masks[0, 0].detach().cpu()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(image)
axes[0].set_title("Input")
axes[0].axis("off")

show_image_with_mask(image, official_mask, axes[1], "Official instance post-process")
show_image_with_mask(image, threshold_query_mask, axes[2], "Selected query threshold")
show_image_with_mask(image, rankseg_query_mask, axes[3], "Selected query RankSEG")

plt.tight_layout()

## 3. Semantic output through the Transformers compatibility layer

SAM3 also returns `outputs.semantic_seg` with shape `(batch, 1, height, width)`. This is the compatible path for `rankseg.transformers.postprocess(...)`: the helper restores a single-channel probability map and lets RankSEG produce a multilabel binary mask.

In [ ]:
official_semantic = processor.post_process_semantic_segmentation(
    outputs,
    target_sizes=inputs["original_sizes"].tolist(),
    threshold=0.5,
)[0].detach().cpu().to(torch.long)

rankseg_semantic = rankseg_transformers_postprocess(
    outputs,
    target_sizes=original_size,
    rankseg_kwargs={"metric": "iou", "solver": "RMA"},
)[0, 0].detach().cpu().to(torch.long)

semantic_difference = (official_semantic != rankseg_semantic).to(torch.long)

print("official semantic mask:", tuple(official_semantic.shape))
print("RankSEG semantic mask:", tuple(rankseg_semantic.shape))
print("different pixels:", int(semantic_difference.sum()))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(image)
axes[0].set_title("Input")
axes[0].axis("off")

show_image_with_mask(image, official_semantic, axes[1], "Official semantic threshold")
show_image_with_mask(image, rankseg_semantic, axes[2], "RankSEG semantic")
show_binary(semantic_difference, axes[3], "Difference map")

plt.tight_layout()

## Compatibility note

This notebook demonstrates the supported SAM3 image semantic path through `outputs.semantic_seg`.

It does not claim complete support for SAM3 video, tracker workflows, pipeline outputs, or a generic SAM3 instance-segmentation adapter. The instance section is a direct notebook-level RankSEG comparison on selected raw mask probabilities.